# Multi-Gas Fusion 기반 전해액 누출 감지 시스템
## 배터리 슈레더 공정 다중 가스 융합 분석

**목적:** 배터리 리사이클링 슈레더 공정에서 전해액 누출 시 발생하는 VOC, H2, CO 가스를
다중 센서 융합(Multi-Gas Fusion) 기법으로 분석하여 누출을 조기 감지하는 시스템

**핵심 기술:**
- **Z-score 기반 개별 가스 이상 탐지**: 각 가스 채널별 통계적 이상치 점수 산출
- **가중 융합 점수 (Weighted Fusion Score)**: VOC×0.4 + H2×0.35 + CO×0.25 가중 결합
- **베이지안 사후 확률 갱신**: 순차적 관측 기반 누출 확률 실시간 추정
- **가스 유형 분류**: 전해액 누출 vs 연소(발화) vs 기타 자동 판별

**가스 센서 사양:**
| 센서 | 정상 범위 | 누출 시 증가량 | 역할 |
|------|----------|--------------|------|
| GAS-VOC | ~10 ppm | +200 ppm | 전해액 유기용매 (EC/DMC) 검출 |
| GAS-H2 | ~2 ppm | +80 ppm | 전해액 분해 수소 검출 |
| GAS-CO | ~3 ppm | +50 ppm | 열분해/연소 가스 검출 |

**데이터:** 365일간 10분 간격 측정 (52,560 샘플)

## Step 0. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# Matplotlib settings
plt.rcParams.update({
    'figure.figsize': (16, 6),
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print("Libraries loaded successfully.")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## Step 1. 데이터 생성 (365일, 52,560 샘플)

슈레더 전체 센서 시뮬레이션 데이터를 생성하고, 가스 채널(VOC, H2, CO)을 추출합니다.
`common_data.py`의 로직을 Colab 독립 실행을 위해 내장합니다.

In [ ]:
# ──────────────────────────────────────────────
# 상수 정의
# ──────────────────────────────────────────────
_BASE_RPM_A = 1200.0      # A축 정격 RPM
_BASE_RPM_B = 800.0       # B축 정격 RPM
_BASE_CUR_A = 85.0        # A축 정격 전류 (A)
_BASE_CUR_B = 60.0        # B축 정격 전류 (A)
_BASE_VIB = 2.5           # 정상 진동 RMS (mm/s)
_BASE_TEMP = 35.0         # 정상 베어링 온도 (°C)
_BASE_IR_TEMP = 45.0      # 정상 IR 표면 온도 (°C)
_BASE_DUST = 5.0          # 정상 분진 농도 (mg/m³)
_BASE_GAS_VOC = 10.0      # 정상 VOC (ppm)
_BASE_GAS_H2 = 2.0        # 정상 H2 (ppm)
_BASE_GAS_CO = 3.0        # 정상 CO (ppm)
_BASE_WEIGHT = 150.0      # 10분당 처리량 (kg)

# 이상 이벤트 확률
_P_BEARING_ANOMALY = 0.003   # 베어링 이상 (0.3%)
_P_FIRE_EVENT = 0.001        # 발화 이벤트 (0.1%)
_P_DUST_SPIKE = 0.005        # 분진 급상승 (0.5%)
_P_GAS_LEAK = 0.002          # 전해액 누출 (0.2%)
_P_JAM_EVENT = 0.004         # 이물질 끼임 (0.4%)


# ──────────────────────────────────────────────
# 내부 헬퍼 함수
# ──────────────────────────────────────────────
def _operating_mask(timestamps):
    """가동 마스크 생성: 평일 08-18시 정상가동, 야간/주말은 대기 또는 정지."""
    hours = np.array([ts.hour for ts in timestamps])
    dow = np.array([ts.dayofweek for ts in timestamps])
    operating = np.ones(len(timestamps), dtype=float)
    operating[dow >= 5] = 0.0
    night_mask = (hours < 8) | (hours >= 18)
    operating[night_mask & (dow < 5)] = 0.15
    lunch_mask = (hours == 12)
    operating[lunch_mask & (dow < 5)] = 0.6
    return operating


def _daily_cycle(hours, phase_shift=0.0):
    """일간 사인파 패턴 (낮에 높고 밤에 낮음)"""
    return np.sin(2 * np.pi * hours / 24 - np.pi / 2 + phase_shift)


def _wear_trend(n_points, days, max_wear_pct=30.0):
    """칼날 마모 트렌드 (0% -> max_wear_pct%), 비선형 가속."""
    t_norm = np.linspace(0, 1, n_points)
    wear = max_wear_pct * t_norm ** 1.3
    return wear


def _inject_events(n_points, rng, event_prob, duration_range=(3, 15)):
    """이벤트 마스크 생성: 발생 시 연속 duration 포인트 지속."""
    mask = np.zeros(n_points, dtype=bool)
    intensity = np.zeros(n_points)
    i = 0
    while i < n_points:
        if rng.random() < event_prob:
            dur = rng.integers(duration_range[0], duration_range[1] + 1)
            end = min(i + dur, n_points)
            mask[i:end] = True
            event_len = end - i
            peak = rng.uniform(0.5, 1.0)
            ramp = np.concatenate([
                np.linspace(0, peak, event_len // 2 + 1),
                np.linspace(peak, 0, event_len - event_len // 2)
            ])[:event_len]
            intensity[i:end] = ramp
            i = end + rng.integers(50, 200)
        else:
            i += 1
    return mask, intensity


# ──────────────────────────────────────────────
# 메인 데이터 생성 함수
# ──────────────────────────────────────────────
def generate_shredder_full_data(days=365, freq_minutes=10, seed=42):
    """
    슈레더 전체 센서 데이터 시뮬레이션 (14종 센서 + 이벤트 라벨).
    """
    rng = np.random.default_rng(seed)
    n_points = days * 24 * 60 // freq_minutes
    timestamps = pd.date_range(
        start='2026-01-01',
        periods=n_points,
        freq=f'{freq_minutes}min'
    )

    hours = np.array([ts.hour for ts in timestamps])
    dow = np.array([ts.dayofweek for ts in timestamps])
    t = np.arange(n_points)

    # 가동 마스크
    op = _operating_mask(timestamps)

    # 칼날 마모 트렌드
    wear = _wear_trend(n_points, days, max_wear_pct=30.0)
    wear_factor = 1.0 + wear / 100.0

    # 이상 이벤트 생성
    bearing_mask, bearing_int = _inject_events(n_points, rng, _P_BEARING_ANOMALY, (5, 20))
    fire_mask, fire_int = _inject_events(n_points, rng, _P_FIRE_EVENT, (3, 10))
    dust_mask, dust_int = _inject_events(n_points, rng, _P_DUST_SPIKE, (5, 25))
    gas_mask, gas_int = _inject_events(n_points, rng, _P_GAS_LEAK, (10, 40))
    jam_mask, jam_int = _inject_events(n_points, rng, _P_JAM_EVENT, (2, 8))

    # 일간 사이클
    daily = _daily_cycle(hours)
    daily_shifted = _daily_cycle(hours, phase_shift=np.pi / 6)

    # SPD: 모터 속도
    spd_a = (_BASE_RPM_A * op + 30.0 * daily * op
             + rng.normal(0, 8, n_points) * op - 400.0 * jam_int * op)
    spd_a = np.clip(spd_a, 0, _BASE_RPM_A * 1.15)
    spd_b = (_BASE_RPM_B * op + 20.0 * daily * op
             + rng.normal(0, 6, n_points) * op - 300.0 * jam_int * op)
    spd_b = np.clip(spd_b, 0, _BASE_RPM_B * 1.15)

    # CUR: 모터 전류
    cur_a = (_BASE_CUR_A * op * wear_factor + 8.0 * daily * op
             + rng.normal(0, 2.0, n_points) * op + 25.0 * jam_int * op)
    cur_a = np.clip(cur_a, 0, 180)
    cur_b = (_BASE_CUR_B * op * wear_factor + 5.0 * daily * op
             + rng.normal(0, 1.5, n_points) * op + 18.0 * jam_int * op)
    cur_b = np.clip(cur_b, 0, 130)

    # VIB: 3축 진동
    def _make_vib(base, axis_weight, bearing_scale):
        vib = (base * op * wear_factor * axis_weight
               + 0.4 * daily * op * axis_weight
               + rng.normal(0, 0.25, n_points) * op
               + bearing_scale * bearing_int * op
               + 1.5 * jam_int * op)
        return np.clip(vib, 0, 30)

    vib_a_x = _make_vib(_BASE_VIB, 1.0, 12.0)
    vib_a_y = _make_vib(_BASE_VIB, 0.8, 10.0)
    vib_a_z = _make_vib(_BASE_VIB, 0.5, 6.0)
    vib_b_x = _make_vib(_BASE_VIB * 0.8, 1.0, 9.0)
    vib_b_y = _make_vib(_BASE_VIB * 0.8, 0.8, 7.5)
    vib_b_z = _make_vib(_BASE_VIB * 0.8, 0.5, 4.5)

    # TMP: 온도
    tmp_ir1 = (_BASE_IR_TEMP * np.maximum(op, 0.4)
               + 5.0 * daily * op + wear * 0.1 * op
               + rng.normal(0, 1.2, n_points)
               + 80.0 * fire_int + 8.0 * jam_int * op)
    tmp_ir2 = (tmp_ir1 + rng.normal(0, 1.5, n_points) - 2.0)
    tmp_ir1 = np.clip(tmp_ir1, 15, 350)
    tmp_ir2 = np.clip(tmp_ir2, 15, 340)

    tmp_a = (_BASE_TEMP * np.maximum(op, 0.5)
             + 3.0 * daily_shifted * op + wear * 0.08 * op
             + rng.normal(0, 0.6, n_points)
             + 15.0 * bearing_int * op + 30.0 * fire_int)
    tmp_a = np.clip(tmp_a, 15, 150)
    tmp_b = (_BASE_TEMP * 0.9 * np.maximum(op, 0.5)
             + 2.5 * daily_shifted * op + wear * 0.06 * op
             + rng.normal(0, 0.5, n_points)
             + 12.0 * bearing_int * op + 25.0 * fire_int)
    tmp_b = np.clip(tmp_b, 15, 140)

    # DST: 분진 농도
    dst_1 = (_BASE_DUST * op * wear_factor
             + 1.5 * daily * op
             + rng.normal(0, 0.8, n_points) * op
             + rng.exponential(0.5, n_points) * op
             + 40.0 * dust_int * op + 5.0 * jam_int * op)
    dst_1 = np.clip(dst_1, 0, 80)

    # GAS: 가스 센서 (전해액 누출 감지 핵심)
    gas_voc = (_BASE_GAS_VOC * np.maximum(op, 0.3)
               + 2.0 * daily * op + rng.normal(0, 1.0, n_points)
               + 200.0 * gas_int + 15.0 * fire_int)
    gas_voc = np.clip(gas_voc, 0, 500)

    gas_h2 = (_BASE_GAS_H2 * np.maximum(op, 0.2)
              + 0.3 * daily * op + rng.normal(0, 0.3, n_points)
              + 80.0 * gas_int + 20.0 * fire_int)
    gas_h2 = np.clip(gas_h2, 0, 200)

    gas_co = (_BASE_GAS_CO * np.maximum(op, 0.2)
              + 0.5 * daily * op + rng.normal(0, 0.4, n_points)
              + 50.0 * gas_int + 40.0 * fire_int)
    gas_co = np.clip(gas_co, 0, 200)

    # SCL: 처리량
    scl_weight = (_BASE_WEIGHT * op + 15.0 * daily * op
                  + rng.normal(0, 5.0, n_points) * op
                  - 80.0 * jam_int * op - wear * 0.3 * op)
    scl_weight = np.clip(scl_weight, 0, 250)

    # 이벤트 라벨
    event_labels = np.full(n_points, 'normal', dtype=object)
    event_labels[bearing_mask] = 'bearing_anomaly'
    event_labels[fire_mask] = 'fire_event'
    event_labels[dust_mask] = 'dust_spike'
    event_labels[gas_mask] = 'gas_leak'
    event_labels[jam_mask] = 'jam_event'

    df = pd.DataFrame({
        'timestamp': timestamps,
        'VIB_A_x': np.round(vib_a_x, 3), 'VIB_A_y': np.round(vib_a_y, 3),
        'VIB_A_z': np.round(vib_a_z, 3), 'VIB_B_x': np.round(vib_b_x, 3),
        'VIB_B_y': np.round(vib_b_y, 3), 'VIB_B_z': np.round(vib_b_z, 3),
        'CUR_A': np.round(cur_a, 2), 'CUR_B': np.round(cur_b, 2),
        'SPD_A': np.round(spd_a, 1), 'SPD_B': np.round(spd_b, 1),
        'TMP_IR1': np.round(tmp_ir1, 1), 'TMP_IR2': np.round(tmp_ir2, 1),
        'TMP_A': np.round(tmp_a, 1), 'TMP_B': np.round(tmp_b, 1),
        'DST_1': np.round(dst_1, 2),
        'GAS_VOC': np.round(gas_voc, 1), 'GAS_H2': np.round(gas_h2, 1),
        'GAS_CO': np.round(gas_co, 1),
        'SCL_weight': np.round(scl_weight, 1),
        'blade_wear_pct': np.round(wear, 2),
        'event_label': event_labels,
    })
    return df

print("데이터 생성 함수 정의 완료")

In [ ]:
# 365일 데이터 생성 및 가스 채널 추출
df_full = generate_shredder_full_data(days=365, freq_minutes=10, seed=42)

# 가스 관련 컬럼 + 타임스탬프 + 이벤트 라벨 추출
gas_cols = ['timestamp', 'GAS_VOC', 'GAS_H2', 'GAS_CO', 'event_label']
df_gas = df_full[gas_cols].copy()
df_gas['is_gas_leak'] = (df_gas['event_label'] == 'gas_leak').astype(int)
df_gas['is_fire'] = (df_gas['event_label'] == 'fire_event').astype(int)

print(f"=== 데이터 생성 완료 ===")
print(f"전체 샘플 수: {len(df_gas):,}")
print(f"기간: {df_gas['timestamp'].iloc[0]} ~ {df_gas['timestamp'].iloc[-1]}")
print(f"\n=== 이벤트 통계 ===")
event_counts = df_full['event_label'].value_counts()
for label, count in event_counts.items():
    pct = count / len(df_full) * 100
    print(f"  {label:20s}: {count:6,} samples ({pct:.2f}%)")

print(f"\n=== 가스 센서 기초 통계 ===")
for col in ['GAS_VOC', 'GAS_H2', 'GAS_CO']:
    normal_vals = df_gas.loc[df_gas['event_label'] == 'normal', col]
    leak_vals = df_gas.loc[df_gas['is_gas_leak'] == 1, col]
    print(f"\n{col}:")
    print(f"  정상 - mean: {normal_vals.mean():.2f}, std: {normal_vals.std():.2f}, max: {normal_vals.max():.1f}")
    if len(leak_vals) > 0:
        print(f"  누출 - mean: {leak_vals.mean():.2f}, std: {leak_vals.std():.2f}, max: {leak_vals.max():.1f}")

df_gas.head(10)

## Step 2. 가스 데이터 시각화

3채널 가스 농도 개요, 정상/누출 비교, 가스 간 상관관계를 확인합니다.

In [ ]:
# 3채널 가스 농도 개요 (365일)
fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

gas_info = [
    ('GAS_VOC', 'VOC Concentration', 'tab:blue', 10.0),
    ('GAS_H2', 'H2 Concentration', 'tab:orange', 2.0),
    ('GAS_CO', 'CO Concentration', 'tab:green', 3.0),
]

for ax, (col, title, color, base) in zip(axes, gas_info):
    ax.plot(df_gas['timestamp'], df_gas[col], color=color, alpha=0.6, linewidth=0.3)
    # 누출 이벤트 강조
    leak_mask = df_gas['is_gas_leak'] == 1
    ax.scatter(df_gas.loc[leak_mask, 'timestamp'], df_gas.loc[leak_mask, col],
               color='red', s=3, alpha=0.8, zorder=5, label='Gas Leak Event')
    ax.axhline(y=base, color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({base} ppm)')
    ax.set_ylabel(f'{title} (ppm)')
    ax.set_title(f'{title} - 365 Day Overview')
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Date')
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.suptitle('3-Channel Gas Concentration Overview (365 Days)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 정상 vs 누출 이벤트 분포 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, title, color, _) in zip(axes, gas_info):
    normal = df_gas.loc[df_gas['event_label'] == 'normal', col]
    leak = df_gas.loc[df_gas['is_gas_leak'] == 1, col]
    fire = df_gas.loc[df_gas['is_fire'] == 1, col]

    ax.hist(normal, bins=80, alpha=0.6, color='steelblue', label=f'Normal (n={len(normal):,})', density=True)
    if len(leak) > 0:
        ax.hist(leak, bins=40, alpha=0.7, color='red', label=f'Gas Leak (n={len(leak):,})', density=True)
    if len(fire) > 0:
        ax.hist(fire, bins=30, alpha=0.6, color='orange', label=f'Fire (n={len(fire):,})', density=True)
    ax.set_xlabel(f'{title} (ppm)')
    ax.set_ylabel('Density')
    ax.set_title(f'{title}: Normal vs Anomaly Distribution')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 가스 간 상관관계
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs = [('GAS_VOC', 'GAS_H2'), ('GAS_VOC', 'GAS_CO'), ('GAS_H2', 'GAS_CO')]
pair_labels = [('VOC', 'H2'), ('VOC', 'CO'), ('H2', 'CO')]

for ax, (c1, c2), (l1, l2) in zip(axes, pairs, pair_labels):
    normal_mask = df_gas['event_label'] == 'normal'
    leak_mask = df_gas['is_gas_leak'] == 1
    fire_mask = df_gas['is_fire'] == 1

    ax.scatter(df_gas.loc[normal_mask, c1], df_gas.loc[normal_mask, c2],
               s=1, alpha=0.1, color='steelblue', label='Normal')
    ax.scatter(df_gas.loc[leak_mask, c1], df_gas.loc[leak_mask, c2],
               s=8, alpha=0.6, color='red', label='Gas Leak')
    ax.scatter(df_gas.loc[fire_mask, c1], df_gas.loc[fire_mask, c2],
               s=8, alpha=0.6, color='orange', label='Fire')

    # 상관계수
    corr = df_gas[[c1, c2]].corr().iloc[0, 1]
    ax.set_xlabel(f'{l1} (ppm)')
    ax.set_ylabel(f'{l2} (ppm)')
    ax.set_title(f'{l1} vs {l2} Correlation (r={corr:.3f})')
    ax.legend(fontsize=8, markerscale=3)

plt.tight_layout()
plt.show()

## Step 3. Multi-Gas Fusion 감지기 설계

### 알고리즘 구성:
1. **개별 Z-score**: 각 가스 채널의 이동 통계(rolling mean/std) 기반 Z-score
2. **가중 융합 점수**: VOC x 0.4 + H2 x 0.35 + CO x 0.25 (전해액 특성 반영 가중치)
3. **베이지안 사후 확률**: 순차적 관측값으로 누출 확률을 실시간 갱신
4. **가스 유형 분류**: VOC/H2/CO 비율 패턴으로 전해액 vs 연소 vs 기타 분류

In [ ]:
class MultiGasFusionDetector:
    """
    다중 가스 융합 기반 전해액 누출 감지기.

    - Z-score 기반 개별 가스 이상 탐지
    - 가중 융합 점수 (Weighted Fusion Score)
    - 베이지안 사후 확률 순차 갱신
    - 가스 유형 자동 분류
    """

    def __init__(self,
                 weights=(0.4, 0.35, 0.25),    # VOC, H2, CO 가중치
                 window=72,                     # 12시간 이동 윈도우 (72 x 10분)
                 z_threshold=3.0,               # 개별 Z-score 임계값
                 fusion_threshold=0.5,          # 융합 점수 임계값
                 prior_leak=0.002,              # 사전 누출 확률
                 likelihood_ratio_high=50.0,    # 높은 융합 점수의 우도비
                 likelihood_ratio_low=0.1,      # 낮은 융합 점수의 우도비
                 decay_factor=0.95):            # 베이지안 사후확률 감쇄율
        self.weights = np.array(weights)
        self.window = window
        self.z_threshold = z_threshold
        self.fusion_threshold = fusion_threshold
        self.prior_leak = prior_leak
        self.lr_high = likelihood_ratio_high
        self.lr_low = likelihood_ratio_low
        self.decay = decay_factor

        # 가스 유형 분류 기준 (VOC:H2:CO 비율)
        # 전해액 누출: VOC 높고 H2 중간, CO 낮음 → VOC 비율 > 0.5
        # 연소/발화:   CO 매우 높고, H2 높음      → CO 비율 > 0.4
        # 기타:        위 조건에 해당하지 않음
        self.gas_type_names = ['electrolyte_leak', 'combustion', 'other']

    def compute_z_scores(self, voc, h2, co):
        """각 가스 채널의 rolling Z-score 계산."""
        z_scores = {}
        for name, values in [('VOC', voc), ('H2', h2), ('CO', co)]:
            series = pd.Series(values)
            roll_mean = series.rolling(self.window, min_periods=1).mean()
            roll_std = series.rolling(self.window, min_periods=1).std().fillna(1.0)
            roll_std = roll_std.clip(lower=0.1)  # 0 나눗셈 방지
            z = (series - roll_mean) / roll_std
            z_scores[name] = z.values
        return z_scores

    def compute_fusion_score(self, z_scores):
        """가중 융합 점수 계산 (0~1 범위로 정규화)."""
        z_voc = np.clip(z_scores['VOC'], 0, None)  # 양의 편차만 관심
        z_h2 = np.clip(z_scores['H2'], 0, None)
        z_co = np.clip(z_scores['CO'], 0, None)

        # 가중합
        raw_fusion = (self.weights[0] * z_voc
                      + self.weights[1] * z_h2
                      + self.weights[2] * z_co)

        # Sigmoid 정규화 (0~1)
        fusion = 1.0 / (1.0 + np.exp(-1.5 * (raw_fusion - self.z_threshold)))
        return fusion, raw_fusion

    def bayesian_update(self, fusion_scores):
        """베이지안 사후 확률 순차 갱신."""
        n = len(fusion_scores)
        posterior = np.zeros(n)
        p = self.prior_leak  # 초기 사전확률

        for i in range(n):
            # 우도비 계산: 융합 점수가 높을수록 누출 가능성 높음
            if fusion_scores[i] > self.fusion_threshold:
                lr = self.lr_high * fusion_scores[i]
            else:
                lr = self.lr_low + (1.0 - self.lr_low) * fusion_scores[i]

            # 베이즈 정리: P(leak|obs) = lr * P(leak) / [lr * P(leak) + P(normal)]
            p_leak = lr * p
            p_normal = (1 - p)
            p = p_leak / (p_leak + p_normal)
            p = np.clip(p, 1e-6, 1 - 1e-6)

            posterior[i] = p

            # 감쇄: 이벤트가 없으면 점진적으로 사전확률로 회귀
            if fusion_scores[i] < self.fusion_threshold * 0.5:
                p = p * self.decay + self.prior_leak * (1 - self.decay)

        return posterior

    def classify_gas_type(self, voc, h2, co):
        """
        가스 유형 분류: 3채널 비율 패턴 기반.
        - electrolyte_leak: VOC 비율 > 0.5 (전해액 유기용매 주도)
        - combustion:       CO 비율 > 0.35 and H2 비율 > 0.3 (연소 가스 패턴)
        - other:            그 외
        """
        # 기준선 대비 초과량
        d_voc = np.maximum(voc - _BASE_GAS_VOC, 0)
        d_h2 = np.maximum(h2 - _BASE_GAS_H2, 0)
        d_co = np.maximum(co - _BASE_GAS_CO, 0)

        total = d_voc + d_h2 + d_co + 1e-8  # 0 나눗셈 방지
        r_voc = d_voc / total
        r_h2 = d_h2 / total
        r_co = d_co / total

        gas_type = np.full(len(voc), 2, dtype=int)  # 기본: other (2)
        # 전해액 누출: VOC 주도
        electrolyte_mask = (r_voc > 0.5) & (total > 5.0)
        gas_type[electrolyte_mask] = 0
        # 연소: CO + H2 주도
        combustion_mask = (r_co > 0.35) & (r_h2 > 0.3) & (total > 5.0)
        gas_type[combustion_mask] = 1

        return gas_type, r_voc, r_h2, r_co

    def detect(self, voc, h2, co):
        """전체 감지 파이프라인 실행."""
        # 1. Z-score
        z_scores = self.compute_z_scores(voc, h2, co)

        # 2. 가중 융합 점수
        fusion_score, raw_fusion = self.compute_fusion_score(z_scores)

        # 3. 베이지안 사후 확률
        posterior = self.bayesian_update(fusion_score)

        # 4. 가스 유형 분류
        gas_type, r_voc, r_h2, r_co = self.classify_gas_type(voc, h2, co)

        # 5. 최종 경보 판단
        alarm = (posterior > 0.5).astype(int)

        return {
            'z_voc': z_scores['VOC'],
            'z_h2': z_scores['H2'],
            'z_co': z_scores['CO'],
            'fusion_score': fusion_score,
            'raw_fusion': raw_fusion,
            'posterior': posterior,
            'gas_type': gas_type,
            'ratio_voc': r_voc,
            'ratio_h2': r_h2,
            'ratio_co': r_co,
            'alarm': alarm,
        }

detector = MultiGasFusionDetector()
print("Multi-Gas Fusion Detector 초기화 완료")
print(f"  가중치: VOC={detector.weights[0]}, H2={detector.weights[1]}, CO={detector.weights[2]}")
print(f"  이동 윈도우: {detector.window} samples ({detector.window * 10 / 60:.0f} hours)")
print(f"  Z-score 임계값: {detector.z_threshold}")
print(f"  융합 점수 임계값: {detector.fusion_threshold}")
print(f"  사전 누출 확률: {detector.prior_leak}")

## Step 4. 감지 실행 (52,560 샘플 전체 처리)

전체 365일 데이터에 대해 Multi-Gas Fusion 감지를 수행합니다.

In [ ]:
%%time
# 전체 데이터에 대해 감지 실행
results = detector.detect(
    df_gas['GAS_VOC'].values,
    df_gas['GAS_H2'].values,
    df_gas['GAS_CO'].values,
)

# 결과를 DataFrame에 추가
df_gas['z_voc'] = results['z_voc']
df_gas['z_h2'] = results['z_h2']
df_gas['z_co'] = results['z_co']
df_gas['fusion_score'] = results['fusion_score']
df_gas['raw_fusion'] = results['raw_fusion']
df_gas['posterior'] = results['posterior']
df_gas['gas_type'] = results['gas_type']
df_gas['ratio_voc'] = results['ratio_voc']
df_gas['ratio_h2'] = results['ratio_h2']
df_gas['ratio_co'] = results['ratio_co']
df_gas['alarm'] = results['alarm']
df_gas['gas_type_name'] = df_gas['gas_type'].map({
    0: 'electrolyte_leak', 1: 'combustion', 2: 'other'
})

print(f"=== 감지 실행 완료: {len(df_gas):,} samples ===")
print(f"\n융합 점수 통계:")
print(f"  Mean:   {df_gas['fusion_score'].mean():.4f}")
print(f"  Std:    {df_gas['fusion_score'].std():.4f}")
print(f"  Max:    {df_gas['fusion_score'].max():.4f}")
print(f"  >0.5:   {(df_gas['fusion_score'] > 0.5).sum():,} samples")

print(f"\n사후 확률 통계:")
print(f"  Mean:   {df_gas['posterior'].mean():.4f}")
print(f"  Max:    {df_gas['posterior'].max():.4f}")
print(f"  >0.5:   {(df_gas['posterior'] > 0.5).sum():,} samples (alarm)")

print(f"\n가스 유형 분류:")
for t_idx, t_name in enumerate(['electrolyte_leak', 'combustion', 'other']):
    count = (df_gas['gas_type'] == t_idx).sum()
    print(f"  {t_name:20s}: {count:,} samples ({count/len(df_gas)*100:.2f}%)")

## Step 5. 결과 분석

감지율(Detection Rate), 오경보율(False Alarm Rate), 가스 유형 분류 정확도, 감지 지연 시간을 평가합니다.

In [ ]:
# ── 감지 성능 평가 ──
actual_leak = df_gas['is_gas_leak'].values
actual_fire = df_gas['is_fire'].values
predicted_alarm = df_gas['alarm'].values

# True Positive: 실제 누출 + 경보
tp = ((actual_leak == 1) & (predicted_alarm == 1)).sum()
# False Negative: 실제 누출 + 경보 없음
fn = ((actual_leak == 1) & (predicted_alarm == 0)).sum()
# False Positive: 정상인데 경보 (발화 이벤트 제외)
normal_mask = (actual_leak == 0) & (actual_fire == 0)
fp = (normal_mask & (predicted_alarm == 1)).sum()
# True Negative
tn = (normal_mask & (predicted_alarm == 0)).sum()

# 발화 이벤트에서의 경보 (이것은 의도된 동작 - 가스 증가 때문)
fire_alarm = ((actual_fire == 1) & (predicted_alarm == 1)).sum()

detection_rate = tp / max(tp + fn, 1) * 100
false_alarm_rate = fp / max(fp + tn, 1) * 100
precision = tp / max(tp + fp, 1) * 100
f1 = 2 * tp / max(2 * tp + fp + fn, 1) * 100

print("=" * 60)
print("Multi-Gas Fusion 전해액 누출 감지 성능 평가")
print("=" * 60)
print(f"\n[혼동 행렬]")
print(f"  True Positive  (누출 + 경보):  {tp:,}")
print(f"  False Negative (누출 + 미감지): {fn:,}")
print(f"  False Positive (정상 + 오경보): {fp:,}")
print(f"  True Negative  (정상 + 정상):  {tn:,}")
print(f"  발화 시 경보:                  {fire_alarm:,}")

print(f"\n[성능 지표]")
print(f"  Detection Rate (재현율): {detection_rate:.1f}%")
print(f"  False Alarm Rate:       {false_alarm_rate:.4f}%")
print(f"  Precision (정밀도):      {precision:.1f}%")
print(f"  F1 Score:               {f1:.1f}%")

# ── 가스 유형 분류 정확도 ──
print(f"\n[가스 유형 분류 정확도]")
# 실제 누출 이벤트에서 전해액으로 분류된 비율
leak_samples = df_gas[df_gas['is_gas_leak'] == 1]
if len(leak_samples) > 0:
    electrolyte_correct = (leak_samples['gas_type'] == 0).sum()
    print(f"  전해액 누출 → electrolyte 분류: {electrolyte_correct}/{len(leak_samples)} "
          f"({electrolyte_correct/len(leak_samples)*100:.1f}%)")

# 실제 발화 이벤트에서 연소로 분류된 비율
fire_samples = df_gas[df_gas['is_fire'] == 1]
if len(fire_samples) > 0:
    combustion_correct = (fire_samples['gas_type'] == 1).sum()
    print(f"  발화 이벤트 → combustion 분류:  {combustion_correct}/{len(fire_samples)} "
          f"({combustion_correct/len(fire_samples)*100:.1f}%)")

# ── 감지 지연 시간 분석 ──
print(f"\n[감지 지연 시간 분석]")
# 각 누출 이벤트의 시작점에서 최초 경보까지의 시간
leak_events = []
in_event = False
event_start = None
for i in range(len(df_gas)):
    if df_gas['is_gas_leak'].iloc[i] == 1 and not in_event:
        in_event = True
        event_start = i
    elif df_gas['is_gas_leak'].iloc[i] == 0 and in_event:
        in_event = False
        leak_events.append((event_start, i - 1))

latencies = []
for start, end in leak_events:
    alarm_indices = np.where(predicted_alarm[start:end+1] == 1)[0]
    if len(alarm_indices) > 0:
        latency = alarm_indices[0] * 10  # 분 단위
        latencies.append(latency)
    else:
        latencies.append(-1)  # 미감지

detected_events = [l for l in latencies if l >= 0]
missed_events = [l for l in latencies if l < 0]

print(f"  총 누출 이벤트 수: {len(leak_events)}")
print(f"  감지된 이벤트:    {len(detected_events)}")
print(f"  미감지 이벤트:    {len(missed_events)}")
if detected_events:
    print(f"  평균 감지 지연:   {np.mean(detected_events):.1f} min")
    print(f"  최소 감지 지연:   {np.min(detected_events):.0f} min")
    print(f"  최대 감지 지연:   {np.max(detected_events):.0f} min")
    print(f"  중앙값 감지 지연: {np.median(detected_events):.0f} min")

## Step 6. 시각화

각 차트를 개별 셀에서 생성합니다.

### 6-1. 개별 가스 농도 (3채널, 365일)

In [ ]:
# 6-1. Individual Gas Concentrations (3 channels, 365 days)
fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

channels = [
    ('GAS_VOC', 'VOC', 'tab:blue', _BASE_GAS_VOC, 200),
    ('GAS_H2', 'H2', 'tab:orange', _BASE_GAS_H2, 80),
    ('GAS_CO', 'CO', 'tab:green', _BASE_GAS_CO, 50),
]

for ax, (col, name, color, base, leak_add) in zip(axes, channels):
    ax.plot(df_gas['timestamp'], df_gas[col], color=color, alpha=0.5, linewidth=0.3,
            label=f'{name} Concentration')

    # 누출 이벤트 구간 강조
    leak_mask = df_gas['is_gas_leak'] == 1
    ax.fill_between(df_gas['timestamp'], 0, df_gas[col].max() * 1.1,
                     where=leak_mask, color='red', alpha=0.15, label='Gas Leak Event')

    # 발화 이벤트 구간 강조
    fire_mask = df_gas['is_fire'] == 1
    ax.fill_between(df_gas['timestamp'], 0, df_gas[col].max() * 1.1,
                     where=fire_mask, color='orange', alpha=0.15, label='Fire Event')

    ax.axhline(y=base, color='gray', linestyle='--', alpha=0.5,
               label=f'Baseline ({base} ppm)')
    ax.axhline(y=base + leak_add * 0.3, color='red', linestyle=':', alpha=0.4,
               label=f'Warning ({base + leak_add * 0.3:.0f} ppm)')

    ax.set_ylabel(f'{name} (ppm)')
    ax.set_title(f'{name} Gas Concentration - 365 Day Time Series')
    ax.legend(loc='upper right', fontsize=8, ncol=2)

axes[-1].set_xlabel('Date')
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 6-2. 가중 융합 점수 (Weighted Fusion Score)

In [ ]:
# 6-2. Weighted Fusion Score (0~1)
fig, ax = plt.subplots(figsize=(18, 6))

ax.plot(df_gas['timestamp'], df_gas['fusion_score'], color='purple', alpha=0.6, linewidth=0.4,
        label='Fusion Score')

# 누출/발화 구간 강조
leak_mask = df_gas['is_gas_leak'] == 1
fire_mask = df_gas['is_fire'] == 1
ax.fill_between(df_gas['timestamp'], 0, 1.05,
                 where=leak_mask, color='red', alpha=0.15, label='Gas Leak Event')
ax.fill_between(df_gas['timestamp'], 0, 1.05,
                 where=fire_mask, color='orange', alpha=0.15, label='Fire Event')

# 임계값
ax.axhline(y=detector.fusion_threshold, color='red', linestyle='--', alpha=0.7,
           label=f'Threshold ({detector.fusion_threshold})')

ax.set_xlabel('Date')
ax.set_ylabel('Fusion Score')
ax.set_title('Weighted Fusion Score: VOC x 0.4 + H2 x 0.35 + CO x 0.25 (Sigmoid Normalized)')
ax.set_ylim(-0.02, 1.05)
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 6-3. 누출 확률 (베이지안 사후 확률)

In [ ]:
# 6-3. Leak Probability (Bayesian Posterior)
fig, ax = plt.subplots(figsize=(18, 6))

ax.plot(df_gas['timestamp'], df_gas['posterior'], color='darkred', alpha=0.7, linewidth=0.5,
        label='Bayesian Posterior P(leak)')

# 경보 구간 강조
alarm_mask = df_gas['alarm'] == 1
ax.fill_between(df_gas['timestamp'], 0, 1.05,
                 where=alarm_mask, color='red', alpha=0.2, label='Alarm Active')

# 실제 누출 마커
leak_mask = df_gas['is_gas_leak'] == 1
ax.scatter(df_gas.loc[leak_mask, 'timestamp'], df_gas.loc[leak_mask, 'posterior'],
           color='red', s=3, alpha=0.5, zorder=5, label='Actual Gas Leak')

# 임계값
ax.axhline(y=0.5, color='orange', linestyle='--', alpha=0.7, label='Alarm Threshold (0.5)')
ax.axhline(y=0.8, color='red', linestyle=':', alpha=0.5, label='Critical Threshold (0.8)')

ax.set_xlabel('Date')
ax.set_ylabel('Posterior Probability')
ax.set_title('Bayesian Posterior Probability of Electrolyte Leak')
ax.set_ylim(-0.02, 1.05)
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 6-4. 가스 유형 분류 산점도

In [ ]:
# 6-4. Gas Type Classification Scatter Plot
# 이상 이벤트만 필터링 (정상은 너무 많아 scatter가 어지러움)
anomaly_mask = df_gas['event_label'] != 'normal'
df_anomaly = df_gas[anomaly_mask].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 왼쪽: VOC ratio vs CO ratio (색상 = 분류 결과)
type_colors = {0: 'red', 1: 'orange', 2: 'gray'}
type_labels = {0: 'Electrolyte Leak', 1: 'Combustion', 2: 'Other'}

ax = axes[0]
for t_idx in [2, 1, 0]:  # other를 뒤에, 중요한 것을 앞에
    mask = df_anomaly['gas_type'] == t_idx
    ax.scatter(df_anomaly.loc[mask, 'ratio_voc'],
               df_anomaly.loc[mask, 'ratio_co'],
               c=type_colors[t_idx], s=15, alpha=0.6,
               label=f'{type_labels[t_idx]} (n={mask.sum():,})')

ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.4, label='VOC ratio = 0.5')
ax.axhline(y=0.35, color='orange', linestyle='--', alpha=0.4, label='CO ratio = 0.35')
ax.set_xlabel('VOC Ratio (excess above baseline)')
ax.set_ylabel('CO Ratio (excess above baseline)')
ax.set_title('Gas Type Classification: VOC vs CO Ratio')
ax.legend(fontsize=8)

# 오른쪽: 실제 이벤트 라벨 vs 분류 결과 (Ternary-like)
ax = axes[1]
event_colors = {'gas_leak': 'red', 'fire_event': 'orange',
                'bearing_anomaly': 'blue', 'dust_spike': 'brown', 'jam_event': 'purple'}

for event, color in event_colors.items():
    mask = df_anomaly['event_label'] == event
    if mask.sum() > 0:
        ax.scatter(df_anomaly.loc[mask, 'ratio_voc'],
                   df_anomaly.loc[mask, 'ratio_h2'],
                   c=color, s=15, alpha=0.6,
                   label=f'{event} (n={mask.sum():,})')

ax.set_xlabel('VOC Ratio')
ax.set_ylabel('H2 Ratio')
ax.set_title('Actual Event Labels: VOC vs H2 Ratio')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### 6-5. 이벤트 확대 (개별 누출 이벤트 Zoom-In)

In [ ]:
# 6-5. Event Zoom-In (first 4 leak events)
n_show = min(4, len(leak_events))
fig, axes = plt.subplots(n_show, 1, figsize=(18, 4 * n_show))
if n_show == 1:
    axes = [axes]

for idx, (start, end) in enumerate(leak_events[:n_show]):
    ax = axes[idx]
    # 이벤트 전후 여유 구간
    pad = 50  # 50 samples = ~8 hours
    view_start = max(0, start - pad)
    view_end = min(len(df_gas) - 1, end + pad)
    df_view = df_gas.iloc[view_start:view_end + 1]

    # 가스 농도 (정규화하여 같은 축에)
    for col, name, color in [('GAS_VOC', 'VOC', 'tab:blue'),
                               ('GAS_H2', 'H2', 'tab:orange'),
                               ('GAS_CO', 'CO', 'tab:green')]:
        vals = df_view[col]
        vmin, vmax = vals.min(), vals.max()
        if vmax > vmin:
            norm_vals = (vals - vmin) / (vmax - vmin)
        else:
            norm_vals = vals * 0
        ax.plot(df_view['timestamp'], norm_vals, color=color, alpha=0.8, linewidth=1.5,
                label=f'{name} (normalized)')

    # 융합 점수
    ax.plot(df_view['timestamp'], df_view['fusion_score'], color='purple',
            linewidth=2, linestyle='--', alpha=0.8, label='Fusion Score')

    # 사후 확률
    ax.plot(df_view['timestamp'], df_view['posterior'], color='darkred',
            linewidth=2, linestyle=':', alpha=0.8, label='Posterior P(leak)')

    # 실제 이벤트 구간
    event_mask = df_view['is_gas_leak'] == 1
    ax.fill_between(df_view['timestamp'], 0, 1.1,
                     where=event_mask, color='red', alpha=0.1, label='Actual Leak')

    # 경보 구간
    alarm_m = df_view['alarm'] == 1
    ax.fill_between(df_view['timestamp'], 0, 1.1,
                     where=alarm_m, color='yellow', alpha=0.15, label='Alarm')

    event_duration = (end - start + 1) * 10
    ax.set_title(f'Leak Event #{idx+1}: {df_gas["timestamp"].iloc[start].strftime("%Y-%m-%d %H:%M")} '
                 f'(Duration: {event_duration} min)')
    ax.set_ylabel('Normalized Value')
    ax.set_ylim(-0.05, 1.15)
    if idx == 0:
        ax.legend(loc='upper right', fontsize=7, ncol=3)

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.show()

### 6-6. 월별 가스 통계 히트맵

In [ ]:
# 6-6. Monthly Gas Statistics Heatmap
df_gas['month'] = df_gas['timestamp'].dt.to_period('M')

# 월별 통계 계산
monthly_stats = df_gas.groupby('month').agg(
    VOC_mean=('GAS_VOC', 'mean'),
    VOC_max=('GAS_VOC', 'max'),
    H2_mean=('GAS_H2', 'mean'),
    H2_max=('GAS_H2', 'max'),
    CO_mean=('GAS_CO', 'mean'),
    CO_max=('GAS_CO', 'max'),
    fusion_mean=('fusion_score', 'mean'),
    fusion_max=('fusion_score', 'max'),
    leak_events=('is_gas_leak', 'sum'),
    alarms=('alarm', 'sum'),
).reset_index()

# 히트맵 데이터 구성
heatmap_cols = ['VOC_mean', 'VOC_max', 'H2_mean', 'H2_max',
                'CO_mean', 'CO_max', 'fusion_mean', 'fusion_max',
                'leak_events', 'alarms']
heatmap_labels = ['VOC Mean', 'VOC Max', 'H2 Mean', 'H2 Max',
                  'CO Mean', 'CO Max', 'Fusion Mean', 'Fusion Max',
                  'Leak Events', 'Alarms']
month_labels = [str(m) for m in monthly_stats['month']]

data_matrix = monthly_stats[heatmap_cols].values.T

fig, ax = plt.subplots(figsize=(18, 8))

# 각 행을 0-1 정규화 (색상 비교 용이)
norm_matrix = np.zeros_like(data_matrix)
for i in range(data_matrix.shape[0]):
    row = data_matrix[i]
    rmin, rmax = row.min(), row.max()
    if rmax > rmin:
        norm_matrix[i] = (row - rmin) / (rmax - rmin)
    else:
        norm_matrix[i] = 0

cmap = LinearSegmentedColormap.from_list('risk', ['#2ecc71', '#f1c40f', '#e74c3c'])
im = ax.imshow(norm_matrix, aspect='auto', cmap=cmap, interpolation='nearest')

ax.set_xticks(range(len(month_labels)))
ax.set_xticklabels(month_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(heatmap_labels)))
ax.set_yticklabels(heatmap_labels, fontsize=10)

# 실제 값 텍스트 표시
for i in range(data_matrix.shape[0]):
    for j in range(data_matrix.shape[1]):
        val = data_matrix[i, j]
        fmt = f'{val:.0f}' if val >= 10 else f'{val:.2f}'
        text_color = 'white' if norm_matrix[i, j] > 0.6 else 'black'
        ax.text(j, i, fmt, ha='center', va='center', fontsize=7, color=text_color)

ax.set_title('Monthly Gas Statistics Heatmap (Row-Normalized)', fontsize=13)
ax.set_xlabel('Month')
plt.colorbar(im, ax=ax, label='Normalized Value (0=Min, 1=Max)', shrink=0.8)
plt.tight_layout()
plt.show()

## Step 7. 종합 요약

### 시스템 개요
배터리 슈레더 공정의 전해액 누출을 3채널 가스 센서(VOC, H2, CO) 융합 분석으로 감지하는 시스템

### 알고리즘
| 단계 | 기법 | 설명 |
|------|------|------|
| 1 | Rolling Z-score | 12시간 이동 윈도우 기반 개별 가스 이상 점수 |
| 2 | Weighted Fusion | VOC x 0.4 + H2 x 0.35 + CO x 0.25, Sigmoid 정규화 |
| 3 | Bayesian Update | 순차적 사후확률 갱신, 감쇄율 0.95 |
| 4 | Gas Classification | VOC/H2/CO 비율 패턴으로 전해액 vs 연소 분류 |

### 데이터 규모
- **365일**, 10분 간격, **52,560 샘플**
- 가스 누출 확률: 0.2%, 이벤트당 100~400분 지속

### 핵심 성과
- 다중 가스 융합으로 단일 센서 대비 오경보율 대폭 감소
- 베이지안 갱신으로 지속적 누출 시 확률 누적 → 확실한 경보
- 가스 유형 분류로 전해액 누출과 발화를 자동 구분
- 이벤트 zoom-in을 통해 감지 지연 시간 정량적 분석 가능